# Network Graph Viewer in a notebook

The whole app runs in the cell output: sidebar, filters, metrics, data
table, exports. Nothing is uploaded anywhere. The graph goes out to the
browser as the app's own workspace format, and the selection and any
edits come back to the kernel.

```sh
uv add network-graph-viewer
```

In [1]:
import pandas as pd

import network_graph_viewer as ngv

ngv.__version__

'0.1.0'

## An edge list

Any two columns will do. `source` and `target` name which is which; everything else rides along as an edge attribute you can style, filter and sort by.

In [2]:
edges = pd.DataFrame(
    [
        {"from": "ana", "to": "ben", "weight": 3, "team": "design"},
        {"from": "ben", "to": "cleo", "weight": 1, "team": "design"},
        {"from": "cleo", "to": "ana", "weight": 4, "team": "research"},
        {"from": "ana", "to": "dev", "weight": 2, "team": "research"},
        {"from": "dev", "to": "eli", "weight": 5, "team": "ops"},
    ]
)
edges

,from,to,weight,team
0,ana,ben,3,design
1,ben,cleo,1,design
2,cleo,ana,4,research
3,ana,dev,2,research
4,dev,eli,5,ops


In [3]:
w = ngv.show(edges, source="from", target="to", color="team")
w

## Reading the graph back

The widget is live. Click a node above, then run the next cell: the kernel sees the selection. Edit a cell in the data table, or compute a metric, and `w.edges` and `w.nodes` come back with the change in them.

Both are `None` until the browser has reported something, so a headless run leaves them empty.

In [4]:
print("selected:", w.selected_node)
w.nodes

selected: None


## Node attributes of their own

Pass a `nodes` table and its rows outlive the edges that named them, so an isolated node still appears. Anything an edge names but the table does not gets a row anyway.

In [5]:
people = pd.DataFrame(
    [
        {"Id": "ana", "dept": "design", "years": 6},
        {"Id": "ben", "dept": "design", "years": 2},
        {"Id": "cleo", "dept": "research", "years": 9},
        {"Id": "zoe", "dept": "ops", "years": 1},
    ]
)

ngv.show(
    edges,
    source="from",
    target="to",
    nodes=people,
    color="dept",
    size="years",
    name="People",
)

## From networkx

A graph brings its node and edge attributes with it.

In [6]:
import networkx as nx

karate = nx.karate_club_graph()
ngv.show(karate, color="club", layout="forceatlas2", height="600px")

## Other shapes of input

A list of `(source, target)` pairs, or a list of dicts, works the same way.

In [7]:
ngv.show([("a", "b"), ("b", "c"), ("c", "a"), ("c", "d")], height="400px")

## Saving

`save()` writes a `.ngv.json` workspace: the tables, the styling, the filters and where everything sat. Open it in the app, or hand it back to `GraphWidget` later.

In [8]:
import pathlib
import tempfile

path = pathlib.Path(tempfile.mkdtemp()) / "team.ngv.json"
w.save(path)
print(path, path.stat().st_size, "bytes")

/tmp/tmpafgvjh4z/team.ngv.json 1968 bytes


## Building the workspace without a widget

`build_workspace` returns the plain dictionary, if you want to write the file from a script or send it somewhere yourself.

In [9]:
workspace = ngv.build_workspace(edges, source="from", target="to")
workspace["doc"]["mapping"], len(workspace["doc"]["nodes"]["rows"])

({'source': 'from', 'target': 'to', 'attrs': ['weight', 'team']}, 5)

## What the cell shows

By default the cell shows the graph and nothing else. Every panel has a
labelled tab on the edge of the stage that opens it, and `panels=` opens
any of them to start with.

In [10]:
ngv.show(
    edges,
    source="from",
    target="to",
    color="team",
    panels=["sidebar", "table"],
    height="620px",
)

## Light and dark

The widget follows the notebook's own colour scheme and keeps following
it, so switching the JupyterLab theme switches the graph too. Pin it with
`theme="light"` or `theme="dark"`, or change it live in the View menu
on the graph.

In [11]:
ngv.show(edges, source="from", target="to", color="team", theme="light", height="420px")